# 가중 CrossEntropyLoss U-Net — Radar 모범답안2 골격으로 DSB 분할 (Colab)

IOAI 2025 **Individual Radar 모범답안2**(weighted-CE U-Net, 0.7963→0.9834)의 코드 골격(`RadarDS`/`DC`/`UNet`/`CrossEntropyLoss(weight=…)`)을 그대로 **Data Science Bowl 2018**(핵 분할)에 적용한다. DSB 이진마스크를 **3클래스(배경/코어/경계)** 로 바꿔 클래스 불균형을 만들고, **가중 교차엔트로피**로 희소 클래스를 살린다.

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "scipy", "pillow", "scikit-image", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 데이터 다운로드 (stage1_train / stage2_test_final)

In [ ]:
import os, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
for fn in ["stage1_train.zip", "stage2_test_final.zip"]:
    api.competition_download_file("data-science-bowl-2018", fn, path=WORK_DIR, quiet=False)
for fn, out in [("stage1_train.zip", "stage1_train"), ("stage2_test_final.zip", "stage2_test_final")]:
    zp = os.path.join(WORK_DIR, fn)
    with zipfile.ZipFile(zp) as zf: zf.extractall(os.path.join(WORK_DIR, out))
    os.remove(zp)
TRAIN_DIR = os.path.join(WORK_DIR, "stage1_train")
TEST_DIR  = os.path.join(WORK_DIR, "stage2_test_final")   # 채점 기준 테스트셋
print("train samples:", len(os.listdir(TRAIN_DIR)), "| test samples:", len(os.listdir(TEST_DIR)))

## 2. 3클래스 라벨맵 만들기 — 배경 / 희소 코어 / 두꺼운 경계
각 핵 인스턴스 마스크를 **3회 침식**해 핵심만 남긴 **코어(class 1)** 를, 원본−코어로 **두꺼운 경계(class 2)** 를 만든다.
코어는 전체의 약 **2%** 에 불과해 *심한 불균형*을 만든다 → 가중 CE 가 필요한 상황을 재현.

In [ ]:
import numpy as np, glob
from PIL import Image
from scipy.ndimage import binary_erosion

SZ = 128
ERODE_ITERS = 3                         # 두껍게 침식 → 코어를 희소하게(불균형 강화)
STRUCT = np.ones((3, 3), dtype=bool)

def load_sample(sample_dir):
    img_path = glob.glob(os.path.join(sample_dir, "images", "*.png"))[0]
    img = np.asarray(Image.open(img_path).convert("RGB").resize((SZ, SZ)), dtype="float32") / 255.0
    lab = np.zeros((SZ, SZ), dtype="int64")          # 0=배경
    for mp in glob.glob(os.path.join(sample_dir, "masks", "*.png")):
        m = np.asarray(Image.open(mp).convert("L").resize((SZ, SZ), Image.NEAREST)) > 0
        if m.sum() == 0:
            continue
        core = binary_erosion(m, STRUCT, iterations=ERODE_ITERS)
        lab[m] = 2                                    # 2=핵 경계 (두꺼운 띠)
        lab[core] = 1                                 # 1=핵 코어 (희소, 코어가 경계를 덮어씀)
    return img.transpose(2, 0, 1), lab

ids = sorted(os.listdir(TRAIN_DIR))
X = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[0] for i in ids])
Y = np.stack([load_sample(os.path.join(TRAIN_DIR, i))[1] for i in ids])
counts = np.bincount(Y.reshape(-1), minlength=3)
print("X:", X.shape, "| Y:", Y.shape)
print("픽셀 비율  배경:%.3f 코어:%.3f 경계:%.3f" % tuple(counts / counts.sum()))

## 3. Dataset & DataLoader (Radar 모범답안2 `RadarDS` 구조 그대로)
인덱스로 `(이미지, 라벨)` 을 돌려주는 `SegDS`(Dataset) + `DataLoader` (Radar 의 `RadarDS`/`DataLoader` 대응).

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class SegDS(Dataset):
    def __init__(self, X, Y, lbl=True): self.X = X; self.Y = Y; self.lbl = lbl
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = torch.tensor(self.X[i])
        if self.lbl: return x, torch.tensor(self.Y[i], dtype=torch.long)
        return x

tr_i, va_i = train_test_split(np.arange(len(X)), test_size=0.1, random_state=42)
train_dl = DataLoader(SegDS(X[tr_i], Y[tr_i]), batch_size=16, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("train:", len(tr_i), "| valid:", len(va_i))

## 4. DC / UNet (Radar 모범답안2 그대로, cout=3)
`DC`(double-conv 블록)와 `UNet`(인코더-디코더). Radar 모범답안2의 `UNet(cin,cout)` 골격 그대로, 여기선 **cin=3(RGB), cout=3(배경/코어/경계)**.

In [ ]:
import torch.nn as nn

class DC(nn.Module):   # Radar 모범답안2 의 double-conv 블록 그대로
    def __init__(s, ci, co):
        super().__init__()
        s.n = nn.Sequential(nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True),
                            nn.Conv2d(co, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(True))
    def forward(s, x): return s.n(x)

class UNet(nn.Module):   # Radar 모범답안2 골격 그대로 (cin=3, cout=3)
    def __init__(s, cin=3, cout=3, b=32):
        super().__init__()
        s.e1 = DC(cin, b); s.e2 = DC(b, b*2); s.e3 = DC(b*2, b*4); s.bo = DC(b*4, b*8); s.pool = nn.MaxPool2d(2)
        s.u3 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2); s.d3 = DC(b*8, b*4)
        s.u2 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2); s.d2 = DC(b*4, b*2)
        s.u1 = nn.ConvTranspose2d(b*2, b, 2, stride=2);   s.d1 = DC(b*2, b); s.o = nn.Conv2d(b, cout, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.pool(e1)); e3 = s.e3(s.pool(e2)); bo = s.bo(s.pool(e3))
        d3 = s.d3(torch.cat([s.u3(bo), e3], 1)); d2 = s.d2(torch.cat([s.u2(d3), e2], 1)); d1 = s.d1(torch.cat([s.u1(d2), e1], 1))
        return s.o(d1)

print('UNet 파라미터:', sum(p.numel() for p in UNet().parameters()))

## 5. 학습 — 가중 CrossEntropyLoss (모범답안2의 핵심 한 줄)
`crit = nn.CrossEntropyLoss(weight=…)` — 배경 1, 희소 비배경 클래스를 강조(Radar 모범답안은 `[1,50,50,50,50]`). argmax 예측으로 검증 per-class IoU 확인.

In [ ]:
EPOCHS = 15
# 역빈도 클래스 가중치 [배경, 코어, 경계] (배경 1 기준, 희소 클래스 강조)
freq = counts / counts.sum(); w = 1.0 / freq; w = w / w.mean()
class_weight = torch.tensor(w, dtype=torch.float32, device=device)
print('클래스 가중치 [배경,코어,경계]:', np.round(w, 2))

torch.manual_seed(0)
model = UNet().to(device)
crit = nn.CrossEntropyLoss(weight=class_weight)   # ★ 핵심: 배경 1, 희소 코어/경계 강조
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for ep in range(EPOCHS):
    model.train(); tot = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step(); tot += loss.item()
    if (ep + 1) % 3 == 0: print(f'epoch {ep+1}/{EPOCHS}  loss {tot/len(train_dl):.4f}')

# 검증 per-class IoU (argmax)
def iou_per_class(pred, gt, n=3):
    out = []
    for c in range(n):
        p, g = (pred == c), (gt == c); u = (p | g).sum()
        out.append(float((p & g).sum()) / float(u) if u > 0 else float('nan'))
    return out
val_dl = DataLoader(SegDS(X[va_i], Y[va_i]), batch_size=16)
model.eval(); preds, gts = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        preds.append(model(xb.to(device)).argmax(1).cpu().numpy()); gts.append(yb.numpy())
pred = np.concatenate(preds); gt = np.concatenate(gts); iou = iou_per_class(pred, gt)
print(f'검증 per-class IoU | 배경 {iou[0]:.3f}  코어 {iou[1]:.3f}  경계 {iou[2]:.3f}  | mIoU {np.nanmean(iou):.3f}')
print('→ 가중 CrossEntropyLoss 가 희소한 코어/경계를 살린다 (Radar 의 weighted-CE 0.79→0.98 효과와 동형).')

## 6. 예측 시각화 (가중 CE 모델)

In [ ]:
import matplotlib.pyplot as plt
model.eval(); j = int(va_i[0])
with torch.no_grad():
    pr = model(torch.from_numpy(X[j:j+1]).to(device)).argmax(1)[0].cpu().numpy()
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(X[j].transpose(1, 2, 0)); ax[0].set_title("Image")
ax[1].imshow(Y[j], vmin=0, vmax=2); ax[1].set_title("GT (0=bg/1=core/2=edge)")
ax[2].imshow(pr, vmin=0, vmax=2); ax[2].set_title("Pred (0/1/2)")
for a in ax: a.axis("off")
plt.show()

## 7. 캐글 제출 — 코어를 마커로 watershed → 인스턴스 RLE
되살린 **코어(`pred==1`)** 의 연결요소를 **watershed 마커**로 쓰고, 핵 영역(`pred>=1`) 안에서 영역을 키워
맞닿은 핵을 **분리**한다(09 의 단순 라벨링 대비 개선). DSB2018 은 인스턴스별 RLE 행을 요구한다.

In [ ]:
import pandas as pd
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.morphology import label

def rle_encode(mask):
    pixels = mask.flatten(order="F")            # DSB2018: 열 우선
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

def instances_from_pred(pr):
    full = pr >= 1                              # 코어+경계 = 핵 전체
    markers = label(pr == 1)                    # 희소 코어 = 마커(서로 떨어져 있어 핵을 가른다)
    if markers.max() == 0:
        return label(full)                      # 코어 없으면 통째로 라벨
    dist = ndi.distance_transform_edt(full)
    return watershed(-dist, markers, mask=full)

test_ids = sorted(os.listdir(TEST_DIR)); rows = []
model.eval()
for sid in test_ids:
    paths = glob.glob(os.path.join(TEST_DIR, sid, "images", "*.png"))
    found = False
    if paths:
        orig = Image.open(paths[0]).convert("RGB"); W, H = orig.size
        x = np.asarray(orig.resize((SZ, SZ)), dtype="float32").transpose(2, 0, 1) / 255.0
        with torch.no_grad():
            pr = model(torch.from_numpy(x[None]).to(device)).argmax(1)[0].cpu().numpy()
        inst = instances_from_pred(pr)
        inst = np.asarray(Image.fromarray(inst.astype("int32")).resize((W, H), Image.NEAREST))
        for k in range(1, int(inst.max()) + 1):
            rle = rle_encode(inst == k)
            if rle:
                rows.append({"ImageId": sid, "EncodedPixels": rle}); found = True
    if not found:
        rows.append({"ImageId": sid, "EncodedPixels": "1 1"})   # ID 누락 방지용 더미

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame(rows, columns=["ImageId", "EncodedPixels"]).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(rows),
      "| unique ids:", pd.DataFrame(rows)["ImageId"].nunique(), "/", len(test_ids))

## 🏁 제출하기
`submission.csv` 를 [Data Science Bowl 2018](https://www.kaggle.com/c/data-science-bowl-2018/submit) 에 제출하세요.

**기법 요약** (IOAI 2025 Radar): 다중클래스 U-Net + **가중 교차엔트로피**(역빈도 클래스 가중)로 *심한 클래스
불균형*을 극복 — 희소 클래스(여기선 핵 코어)의 IoU 를 살려 mIoU 를 끌어올린다. Radar 모범답안2 의 0.7963→0.9834
개선과 동형. 더 끌어올리려면: 가중치 스케일 조정(median-frequency balancing), CE+Dice 혼합,
침식 두께(ERODE_ITERS)·해상도(SZ)↑, 증강(flip/rotate).